# Random Forest + SVM — Laptop Buying Classification

Same dataset, two classification algorithms.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


In [ ]:
df = pd.read_csv(
    "/content/laptop_buying_data.csv",
    na_values=["", " ", "NA", "N/A", "null", "NULL", "None", "?"]
)
df.head()


In [ ]:
print("Shape:", df.shape)
print("\nMissing Values:")
print(df.isnull().sum())
print("\nDuplicate Rows:", df.duplicated().sum())


In [ ]:
df = df.drop_duplicates()
print("Shape after removing duplicates:", df.shape)


In [ ]:
print(df["Buys Laptop"].value_counts())

df["Buys Laptop"].value_counts().plot(
    kind="bar", title="Laptop Buying Distribution"
)
plt.xlabel("Buys Laptop")
plt.ylabel("Number of People")
plt.show()


In [ ]:
X = df.drop("Buys Laptop", axis=1)
y = df["Buys Laptop"]

numeric_features = ["Age", "Credit Score"]
categorical_features = ["Income", "Student", "Location"]


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


## Random Forest


In [ ]:
rf_numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

rf_categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])


In [ ]:
rf_preprocessor = ColumnTransformer([
    ("num", rf_numeric_pipeline, numeric_features),
    ("cat", rf_categorical_pipeline, categorical_features)
])


In [ ]:
rf_model = Pipeline([
    ("preprocessor", rf_preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200, max_depth=6, random_state=42
    ))
])


In [ ]:
rf_model.fit(X_train, y_train)


In [ ]:
rf_pred = rf_model.predict(X_test)
print(rf_pred)


In [ ]:
rf_accuracy = accuracy_score(y_test, rf_pred)
print("Random Forest Accuracy:", round(rf_accuracy, 2))
print("\nClassification Report:")
print(classification_report(y_test, rf_pred))


In [ ]:
rf_cm = confusion_matrix(y_test, rf_pred)
print(rf_cm)


In [ ]:
plt.imshow(rf_cm)
plt.title("Random Forest - Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks([0, 1], ["No", "Yes"])
plt.yticks([0, 1], ["No", "Yes"])

for i in range(2):
    for j in range(2):
        plt.text(j, i, rf_cm[i, j], ha="center", va="center")

plt.colorbar()
plt.show()


In [ ]:
rf_cv = cross_val_score(
    rf_model, X_train, y_train, cv=5, scoring="accuracy"
)

print("Random Forest CV Scores:", rf_cv)
print("Mean CV Accuracy:", round(rf_cv.mean(), 2))


In [ ]:
feature_names = rf_model.named_steps["preprocessor"].get_feature_names_out()
importances = rf_model.named_steps["classifier"].feature_importances_

rf_importance = pd.Series(importances, index=feature_names)
print(rf_importance.sort_values(ascending=False))


In [ ]:
rf_importance.sort_values().plot(
    kind="barh", figsize=(8, 6),
    title="Random Forest Feature Importance"
)
plt.xlabel("Importance")
plt.show()


## Support Vector Machine (SVM)


In [ ]:
svm_numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

svm_categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])


In [ ]:
svm_preprocessor = ColumnTransformer([
    ("num", svm_numeric_pipeline, numeric_features),
    ("cat", svm_categorical_pipeline, categorical_features)
])


In [ ]:
svm_model = Pipeline([
    ("preprocessor", svm_preprocessor),
    ("classifier", SVC(
        kernel="rbf", C=1.0, gamma="scale"
    ))
])


In [ ]:
svm_model.fit(X_train, y_train)


In [ ]:
svm_pred = svm_model.predict(X_test)
print(svm_pred)


In [ ]:
svm_accuracy = accuracy_score(y_test, svm_pred)
print("SVM Accuracy:", round(svm_accuracy, 2))
print("\nClassification Report:")
print(classification_report(y_test, svm_pred))


In [ ]:
svm_cm = confusion_matrix(y_test, svm_pred)
print(svm_cm)


In [ ]:
plt.imshow(svm_cm)
plt.title("SVM - Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks([0, 1], ["No", "Yes"])
plt.yticks([0, 1], ["No", "Yes"])

for i in range(2):
    for j in range(2):
        plt.text(j, i, svm_cm[i, j], ha="center", va="center")

plt.colorbar()
plt.show()


In [ ]:
svm_cv = cross_val_score(
    svm_model, X_train, y_train, cv=5, scoring="accuracy"
)

print("SVM CV Scores:", svm_cv)
print("Mean CV Accuracy:", round(svm_cv.mean(), 2))


## Compare Both Models


In [ ]:
comparison = pd.DataFrame({
    "Model": ["Random Forest", "SVM"],
    "Test Accuracy": [rf_accuracy, svm_accuracy],
    "CV Accuracy": [rf_cv.mean(), svm_cv.mean()]
})

comparison


In [ ]:
new_person = pd.DataFrame({
    "Age": [26],
    "Income": ["Medium"],
    "Student": ["Yes"],
    "Credit Score": [710],
    "Location": ["Urban"]
})

rf_prediction = rf_model.predict(new_person)[0]
svm_prediction = svm_model.predict(new_person)[0]

print("Random Forest Prediction:", rf_prediction)
print("SVM Prediction:", svm_prediction)


In [ ]:
svm_score = svm_model.decision_function(new_person)[0]

print("SVM Prediction:", svm_prediction)
print("SVM Decision Score:", round(svm_score, 3))


### Key Difference

**Random Forest:** many decision trees learn patterns and vote.

**SVM:** learns a maximum-margin decision boundary.

**Preprocessing:** Random Forest generally does not need scaling; SVM benefits from scaling numerical features.
